# .lst から Rietveld 結果を直接集計する notebook

`2_Program/compile_rietveld_from_lst.py` を import して、`.lst` から `Chem_df`, `Param_df`, `Goal_df` 相当の表を作り、既存 notebook と同じ5つの Excel を保存します。

必要な環境: `pandas`, `openpyxl`, `pymatgen` が入った Python kernel。

In [1]:
from pathlib import Path
import importlib.util
import sys


# notebook を repo root から開いた場合、2_Program から開いた場合の両方に対応する。
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "2_Program":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPT_PATH = PROJECT_ROOT / "2_Program" / "compile_rietveld_from_lst.py"

# 通常の import ではファイル名ベースの module として読みにくいため、path から明示的に読み込む。
spec = importlib.util.spec_from_file_location("compile_rietveld_from_lst", SCRIPT_PATH)
rietveld = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = rietveld
spec.loader.exec_module(rietveld)

print(f"Loaded: {SCRIPT_PATH}")


Loaded: /Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/2_Program/compile_rietveld_from_lst.py


## 設定

- 全件実行する場合は `SAMPLE = None`
- 1ファイルだけ確認する場合は `SAMPLE` に `.lst` の path を入れる
- `SAVE_INTERMEDIATE_CSV = True` にすると、確認用の `Chem_from_lst.csv`, `Param_from_lst.csv`, `Goal_from_lst.csv` も保存する

In [2]:
INPUT_ROOT = PROJECT_ROOT / "0_Data" / "1_Rietveld"
OUTPUT_DIR = PROJECT_ROOT / "0_Data" / "2_Compiled_Data"

# 全件実行: None
# 1ファイル確認例: INPUT_ROOT / "SPring-8" / "Mafic" / "SP-MTN-43" / "SP-MTN-43.lst"
SAMPLE = None

# OLD フォルダを含める場合だけ True にする。
INCLUDE_OLD = False

# 中間 CSV を保存したい場合だけ True にする。
SAVE_INTERMEDIATE_CSV = False

INPUT_ROOT, OUTPUT_DIR, SAMPLE


(PosixPath('/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_Data/1_Rietveld'),
 PosixPath('/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_Data/2_Compiled_Data'),
 None)

## 実行

下のセルを実行すると、`.lst` を読み、5つの Excel を `OUTPUT_DIR` に保存します。

In [3]:
input_root = INPUT_ROOT.resolve()
output_dir = OUTPUT_DIR.resolve()

if SAMPLE is None:
    lst_paths = [path.resolve() for path in rietveld.find_lst_files(input_root, INCLUDE_OLD)]
else:
    sample_path = Path(SAMPLE).resolve()
    lst_paths = [sample_path]
    if not sample_path.is_relative_to(input_root):
        input_root = rietveld.common_parent(input_root, sample_path)

if not lst_paths:
    raise ValueError("No .lst files found. INPUT_ROOT と SAMPLE を確認してください。")

parsed_files = [rietveld.parse_lst(path) for path in lst_paths]
Chem_df, Param_df, Goal_df = rietveld.build_rows(parsed_files, input_root)
rietveld.write_outputs(Chem_df, Param_df, Goal_df, output_dir)

if SAVE_INTERMEDIATE_CSV:
    Chem_df.to_csv(output_dir / "Chem_from_lst.csv", index=False)
    Param_df.to_csv(output_dir / "Param_from_lst.csv", index=False)
    Goal_df.to_csv(output_dir / "Goal_from_lst.csv", index=False)

print(f"Compiled {len(lst_paths)} .lst file(s)")
print(f"Output: {output_dir}")


Compiled 97 .lst file(s)
Output: /Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_Data/2_Compiled_Data


## 出力確認

In [4]:
output_files = [
    "pivot_chem_df.xlsx",
    "pivot_bulk_comp_df.xlsx",
    "Param_compile_df.xlsx",
    "pivot_error_df.xlsx",
    "pivot_bulk_mode_df.xlsx",
]

for file_name in output_files:
    path = output_dir / file_name
    print(f"{file_name}: {'OK' if path.exists() else 'MISSING'}")

display(Chem_df.head())
display(Param_df.head())
display(Goal_df.head())


pivot_chem_df.xlsx: OK
pivot_bulk_comp_df.xlsx: OK
Param_compile_df.xlsx: OK
pivot_error_df.xlsx: OK
pivot_bulk_mode_df.xlsx: OK


,File,Sample,Sample ID,Instrument,Rock_Type,Phase,Phase Quantity,Oxide,Quantity (wt-%)
0,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,104_MTN22051304_Theta_2-Theta,104_MTN22051304_Theta_2-Theta.txt Data 1,Laboratory,Mafic,SiO2p3221,0.03557,SiO2,100.000000
1,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,104_MTN22051304_Theta_2-Theta,104_MTN22051304_Theta_2-Theta.txt Data 1,Laboratory,Mafic,MUSCOVITE_2M1,0.11520,Al2O3,37.820826
2,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,104_MTN22051304_Theta_2-Theta,104_MTN22051304_Theta_2-Theta.txt Data 1,Laboratory,Mafic,MUSCOVITE_2M1,0.11520,Fe2O3,4.710557
3,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,104_MTN22051304_Theta_2-Theta,104_MTN22051304_Theta_2-Theta.txt Data 1,Laboratory,Mafic,MUSCOVITE_2M1,0.11520,K2O,9.349299
4,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,104_MTN22051304_Theta_2-Theta,104_MTN22051304_Theta_2-Theta.txt Data 1,Laboratory,Mafic,MUSCOVITE_2M1,0.11520,SiO2,48.119317


,File,Sample,Sample ID,Instrument,Rock_Type,Phase,Refined Composition,A,ESD(A),B,...,C,ESD(C),ALPHA,ESD(ALPHA),BETA,ESD(BETA),GAMMA,ESD(GAMMA),"GrainSize(1,1,1)","ESD(GrainSize(1,1,1))"
0,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,104_MTN22051304_Theta_2-Theta,104_MTN22051304_Theta_2-Theta.txt Data 1,Laboratory,Mafic,SiO2p3221,SI3 O6,0.491247,0.000050,NaN,...,0.540680,0.000100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,104_MTN22051304_Theta_2-Theta,104_MTN22051304_Theta_2-Theta.txt Data 1,Laboratory,Mafic,MUSCOVITE_2M1,AL11.1160 FE0.8840 K2.9744 SI12 O48,0.522910,0.000120,0.907100,...,1.997202,0.000072,NaN,NaN,95.8180,0.0100,NaN,NaN,NaN,NaN
2,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,104_MTN22051304_Theta_2-Theta,104_MTN22051304_Theta_2-Theta.txt Data 1,Laboratory,Mafic,Titanite,CA4 SI4 TI4 O20,0.743450,0.000190,0.862960,...,0.706640,0.000160,NaN,NaN,126.2690,0.0180,NaN,NaN,NaN,NaN
3,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,104_MTN22051304_Theta_2-Theta,104_MTN22051304_Theta_2-Theta.txt Data 1,Laboratory,Mafic,Fluorapatite,CA10 F2 P6 O24,0.930333,NaN,NaN,...,0.694698,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,104_MTN22051304_Theta_2-Theta,104_MTN22051304_Theta_2-Theta.txt Data 1,Laboratory,Mafic,Epidote,AL4.0956 CA4 FE1.9044 SI6 O26,0.890420,0.000032,0.564557,...,1.016945,0.000037,NaN,NaN,115.3611,0.0029,NaN,NaN,NaN,NaN


,File,Sample,Sample ID,Instrument,Rock_Type,"Parameter, Goal",Value,ESD
0,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,104_MTN22051304_Theta_2-Theta,104_MTN22051304_Theta_2-Theta.txt Data 1,Laboratory,Mafic,QSiO2p3221,0.03557,0.00075
1,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,104_MTN22051304_Theta_2-Theta,104_MTN22051304_Theta_2-Theta.txt Data 1,Laboratory,Mafic,Qmusc2m1,0.11520,0.00180
2,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,104_MTN22051304_Theta_2-Theta,104_MTN22051304_Theta_2-Theta.txt Data 1,Laboratory,Mafic,Qtitanite,0.05360,0.00110
3,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,104_MTN22051304_Theta_2-Theta,104_MTN22051304_Theta_2-Theta.txt Data 1,Laboratory,Mafic,QFluorapatite,0.00569,0.00089
4,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,104_MTN22051304_Theta_2-Theta,104_MTN22051304_Theta_2-Theta.txt Data 1,Laboratory,Mafic,QEpidote,0.22130,0.00180
